In [1]:
# ╔══════════════════════════════════════════════════════════╗
# ║         CELL 1 — Cài thư viện (chạy 1 lần)           ║
# ╚══════════════════════════════════════════════════════════╝
!pip install PyMuPDF pdfplumber pdf2image pytesseract -q
!apt-get install -y tesseract-ocr tesseract-ocr-vie poppler-utils -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 62.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 90.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 96.0 MB/s eta 0:00:00:00:01
Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  poppler-utils tesseract-ocr-vie
0 upgraded, 2 newly installed, 0 to remove and 133 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,243 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 

In [2]:
# ╔══════════════════════════════════════════════════════════╗
# ║              CELL 2 — Import                            ║
# ╚══════════════════════════════════════════════════════════╝
 
import os
import time
import unicodedata
from dataclasses import dataclass, field
from typing import Optional
 
import fitz          # PyMuPDF
import pdfplumber
import pytesseract
from pdf2image import convert_from_path

In [3]:
# ╔══════════════════════════════════════════════════════════╗
# ║              CELL 3 — Cấu hình                          ║
# ╚══════════════════════════════════════════════════════════╝
 
@dataclass
class Config:
    # ── Phân loại ──────────────────────────────────────────
    # Số ký tự trung bình tối thiểu/trang để coi là có text
    min_avg_chars_per_page: int   = 100
    # Tỷ lệ tối thiểu số trang phải có text
    min_text_page_ratio:    float = 0.5
    # Tỷ lệ diện tích ảnh/trang vượt ngưỡng → nghi là scan
    image_area_ratio_threshold: float = 0.8
    # Tỷ lệ ký tự rác tối đa cho phép
    max_garbage_char_ratio: float = 0.1
    # Số trang tối đa phân tích khi classify (tránh chậm)
    max_pages_to_analyze:   int   = 20
 
    # ── Đọc ────────────────────────────────────────────────
    # Ngôn ngữ OCR: "vie" | "eng" | "vie+eng"
    ocr_language:    str          = "vie+eng"
    # DPI rasterize khi OCR (cao hơn = chính xác hơn, chậm hơn)
    ocr_dpi:         int          = 300
    # Trích xuất bảng khi đọc direct
    extract_tables:  bool         = True
    # Giữ layout gốc (khoảng trắng, cột)
    preserve_layout: bool         = True
    # Số trang tối đa cần đọc (None = đọc tất cả)
    max_pages:       Optional[int] = None
 

In [4]:
# ╔══════════════════════════════════════════════════════════╗
# ║          CELL 4 — Dataclass kết quả                     ║
# ╚══════════════════════════════════════════════════════════╝
 
@dataclass
class ClassifyResult:
    """Kết quả phân loại một file PDF."""
    file:               str
    classification:     str    # "No OCR needed" | "OCR required"
    confidence:         str    # "high" | "medium" | "low"
    total_pages:        int
    pages_analyzed:     int
    avg_chars_per_page: float
    text_page_ratio:    float
    has_embedded_fonts: bool
    image_dom_pages:    int
    garbage_char_ratio: float
    signals:            list   = field(default_factory=list)
    error:              Optional[str] = None
 
    def __str__(self) -> str:
        if self.error:
            return f"[LỖI] {self.file}: {self.error}"
        icon = "✅" if self.classification == "No OCR needed" else "🔴"
        lines = [
            "─" * 55,
            f"  File       : {os.path.basename(self.file)}",
            f"  Kết quả    : {icon} {self.classification}",
            f"  Độ tin cậy : {self.confidence}",
            "─" * 55,
            f"  Tổng trang          : {self.total_pages} (phân tích: {self.pages_analyzed})",
            f"  Ký tự TB/trang      : {self.avg_chars_per_page:.1f}",
            f"  Tỷ lệ trang có text : {self.text_page_ratio*100:.1f}%",
            f"  Font nhúng          : {'Có' if self.has_embedded_fonts else 'Không'}",
            f"  Trang bị ảnh phủ    : {self.image_dom_pages}",
            f"  Tỷ lệ ký tự rác     : {self.garbage_char_ratio*100:.1f}%",
            "─" * 55,
        ]
        for s in self.signals:
            lines.append(f"  • {s}")
        lines.append("─" * 55)
        return "\n".join(lines)
 
 
@dataclass
class PageContent:
    """Nội dung một trang PDF sau khi đọc."""
    page_number: int
    text:        str
    tables:      list   # [[row, ...], ...]  — rỗng nếu OCR
    method:      str    # "direct" | "ocr"
    char_count:  int
 
    def __str__(self) -> str:
        lines = [f"\n{'━'*40}",
                 f"  Trang {self.page_number}  [{self.method.upper()}]  ({self.char_count:,} ký tự)",
                 f"{'━'*40}"]
        if self.text:
            lines.append(self.text)
        for i, table in enumerate(self.tables, 1):
            lines.append(f"\n  [Bảng {i}]")
            for row in table:
                lines.append("  | " + " | ".join(str(c or "").strip() for c in row))
        return "\n".join(lines)
 
 
@dataclass
class ReadResult:
    """Kết quả đọc toàn bộ một file PDF."""
    file:             str
    classification:   str
    confidence:       str
    total_pages:      int
    pages_read:       int
    method:           str    # "direct" | "ocr"
    pages:            list   # list[PageContent]
    elapsed_seconds:  float
    error:            Optional[str] = None
 
    @property
    def full_text(self) -> str:
        return "\n\n".join(p.text for p in self.pages if p.text)
 
    @property
    def all_tables(self) -> list:
        out = []
        for p in self.pages:
            out.extend(p.tables)
        return out
 
    def save_text(self, path: str) -> None:
        with open(path, "w", encoding="utf-8") as f:
            f.write(f"File     : {self.file}\n")
            f.write(f"Phương pháp: {self.method}\n")
            f.write("=" * 55 + "\n")
            for p in self.pages:
                f.write(str(p) + "\n")
        print(f"✅ Đã lưu → {path}")
 
    def __str__(self) -> str:
        total_chars = sum(p.char_count for p in self.pages)
        table_count = sum(len(p.tables) for p in self.pages)
        lines = [
            "═" * 55,
            f"  File         : {os.path.basename(self.file)}",
            f"  Phân loại    : {self.classification} ({self.confidence})",
            f"  Phương pháp  : {self.method.upper()}",
            f"  Trang đọc    : {self.pages_read}/{self.total_pages}",
            f"  Tổng ký tự   : {total_chars:,}",
            f"  Bảng tìm được: {table_count}",
            f"  Thời gian    : {self.elapsed_seconds:.1f}s",
            "═" * 55,
        ]
        if self.error:
            lines.append(f"  ⚠️  Lỗi: {self.error}")
        return "\n".join(lines)
 
 

In [5]:
# ╔══════════════════════════════════════════════════════════╗
# ║          CELL 5 — Hàm phụ trợ (nội bộ)                 ║
# ╚══════════════════════════════════════════════════════════╝
 
def _garbage_char_count(text: str) -> int:
    """Đếm ký tự không in được / Unicode lỗi."""
    bad  = {"Cc", "Cs", "Co", "Cn"}
    tofu = {"\u25a1", "\u2610", "\ufffe", "\uffff", "\ufffd"}
    return sum(1 for ch in text
               if unicodedata.category(ch) in bad or ch in tofu)
 
 
def _image_area_ratio(page: fitz.Page) -> float:
    """Tỷ lệ diện tích ảnh nhúng / diện tích trang (0.0–1.0)."""
    page_area = page.rect.width * page.rect.height
    if page_area == 0:
        return 0.0
    img_area = sum(
        max(i["bbox"][2] - i["bbox"][0], 0) * max(i["bbox"][3] - i["bbox"][1], 0)
        for i in page.get_image_info(xrefs=True)
        if "bbox" in i
    )
    return min(img_area / page_area, 1.0)
 
 
def _fonts_ok(doc: fitz.Document, n: int) -> bool:
    """True nếu < 30% trang có font không nhúng."""
    bad_pages = 0
    checked   = min(len(doc), n)
    for i in range(checked):
        for font in doc.load_page(i).get_fonts(full=True):
            # font = (xref, ext, type, basefont, name, encoding, ref)
            if font[1] == "" and font[2] not in ("Type3", ""):
                bad_pages += 1
                break
    return (bad_pages / checked) < 0.3 if checked else True
 
 
def _is_legacy_vietnamese_font(text: str) -> bool:
    """
    Phát hiện PDF dùng font encoding cũ kiểu TCVN3 / VNI / ABC.
    Các font này map ký tự Latin Extended (U+00A0–U+00FF) sang chữ Việt,
    khiến PyMuPDF đọc ra ký tự lạ như: µ ¹ ® © ¬ ­ Ö × ê ë...
    thay vì Unicode tiếng Việt chuẩn.
 
    Dấu hiệu nhận biết:
      - Tỷ lệ Latin Extended cao bất thường (> 8%)
      - Xuất hiện các ký tự đặc trưng của TCVN3 bị nhầm
    """
    if len(text) < 50:
        return False
 
    # Ký tự Latin Extended bị nhầm phổ biến trong TCVN3/VNI
    tcvn3_markers = set("µ¸¹²³¾»¼½¿ÖÊËÌÍÎÏÐÑÒÓÔÕÙÚÛÜÝàáâãäåæèéêëìíîïðñòóôõöùúûüý")
    latin_ext   = sum(1 for ch in text if "\u00A0" <= ch <= "\u00FF")
    marker_hits = sum(1 for ch in text if ch in tcvn3_markers)
 
    ratio_latin  = latin_ext   / len(text)
    ratio_marker = marker_hits / len(text)
 
    # Cả 2 tiêu chí đều vượt ngưỡng → chắc chắn là font cũ
    return ratio_latin > 0.08 and ratio_marker > 0.03
 
 

In [6]:
# ╔══════════════════════════════════════════════════════════╗
# ║          CELL 6 — Phân loại PDF                         ║
# ╚══════════════════════════════════════════════════════════╝
 
def classify_pdf(pdf_path: str, cfg: Config) -> ClassifyResult:
    """
    Phân loại PDF dựa trên 6 tín hiệu:
      1. Mật độ ký tự/trang
      2. Tỷ lệ trang có text
      3. Trang bị ảnh phủ
      4. Font không nhúng
      5. Tỷ lệ ký tự rác
      6. Font encoding cũ (TCVN3/VNI/ABC)  ← mới
    """
    # ── Kiểm tra file ──
    if not os.path.exists(pdf_path):
        return ClassifyResult(
            file=pdf_path, classification="OCR required", confidence="low",
            total_pages=0, pages_analyzed=0, avg_chars_per_page=0,
            text_page_ratio=0, has_embedded_fonts=False,
            image_dom_pages=0, garbage_char_ratio=0,
            error=f"Không tìm thấy file: {pdf_path}",
        )
 
    try:
        doc = fitz.open(pdf_path)
    except Exception as e:
        return ClassifyResult(
            file=pdf_path, classification="OCR required", confidence="low",
            total_pages=0, pages_analyzed=0, avg_chars_per_page=0,
            text_page_ratio=0, has_embedded_fonts=False,
            image_dom_pages=0, garbage_char_ratio=0,
            error=f"Không mở được file: {e}",
        )
 
    total_pages = len(doc)
    if total_pages == 0:
        doc.close()
        return ClassifyResult(
            file=pdf_path, classification="OCR required", confidence="low",
            total_pages=0, pages_analyzed=0, avg_chars_per_page=0,
            text_page_ratio=0, has_embedded_fonts=False,
            image_dom_pages=0, garbage_char_ratio=0,
            error="PDF không có trang nào.",
        )
 
    n = min(total_pages, cfg.max_pages_to_analyze)
 
    # ── Thu thập số liệu từng trang ──
    total_chars, total_garbage, pages_with_text, img_dom = 0, 0, 0, 0
    sample_text = ""   # lấy mẫu để kiểm tra encoding
 
    for i in range(n):
        page  = doc.load_page(i)
        text  = page.get_text().strip()
        total_chars   += len(text)
        total_garbage += _garbage_char_count(text)
        if len(text) >= cfg.min_avg_chars_per_page:
            pages_with_text += 1
        if _image_area_ratio(page) >= cfg.image_area_ratio_threshold:
            img_dom += 1
        if i < 3:                  # chỉ cần mẫu 3 trang đầu
            sample_text += text
 
    avg_chars     = total_chars / n
    text_ratio    = pages_with_text / n
    garbage_ratio = total_garbage / max(total_chars, 1)
    fonts_ok      = _fonts_ok(doc, n)
    is_legacy     = _is_legacy_vietnamese_font(sample_text)
    doc.close()
 
    # ── Đánh giá từng tín hiệu ──
    bad, good = [], []
 
    # Tín hiệu 1 — Mật độ ký tự
    if avg_chars < cfg.min_avg_chars_per_page:
        bad.append(f"Ký tự TB/trang thấp ({avg_chars:.0f} < {cfg.min_avg_chars_per_page}).")
    else:
        good.append(f"Ký tự TB/trang đủ lớn ({avg_chars:.0f}).")
 
    # Tín hiệu 2 — Tỷ lệ trang có text
    if text_ratio < cfg.min_text_page_ratio:
        bad.append(f"Chỉ {text_ratio*100:.0f}% trang có văn bản (ngưỡng {cfg.min_text_page_ratio*100:.0f}%).")
    else:
        good.append(f"{text_ratio*100:.0f}% trang có văn bản.")
 
    # Tín hiệu 3 — Trang bị ảnh phủ
    if img_dom / n > 0.5:
        bad.append(f"{img_dom}/{n} trang bị ảnh phủ ≥ {cfg.image_area_ratio_threshold*100:.0f}% diện tích.")
    elif img_dom > 0:
        good.append(f"Có {img_dom} trang ảnh lớn nhưng không chiếm đa số.")
 
    # Tín hiệu 4 — Font không nhúng
    if not fonts_ok:
        bad.append("Nhiều font không nhúng → ký tự có thể bị lỗi.")
    else:
        good.append("Font nhúng đầy đủ.")
 
    # Tín hiệu 5 — Ký tự rác
    if garbage_ratio > cfg.max_garbage_char_ratio:
        bad.append(f"Ký tự rác cao ({garbage_ratio*100:.1f}%) → text bị mã hoá sai.")
    else:
        good.append(f"Ký tự rác thấp ({garbage_ratio*100:.1f}%).")
 
    # Tín hiệu 6 — Font encoding cũ TCVN3/VNI/ABC  ← mới
    if is_legacy:
        bad.append("Phát hiện font encoding cũ (TCVN3/VNI/ABC) → text bị sai ký tự, cần OCR.")
    else:
        good.append("Encoding Unicode chuẩn, text đọc được bình thường.")
 
    # ── Quyết định ──
    # Tín hiệu 6 (legacy font) đủ nghiêm trọng để tự quyết định luôn
    if is_legacy:
        cls, conf = "OCR required", "high"
    elif len(bad) >= 3:
        cls, conf = "OCR required", "high"
    elif len(bad) == 2:
        cls, conf = "OCR required", "medium"
    elif len(bad) == 1:
        cls, conf = "OCR required", "low"
    else:
        cls, conf = "No OCR needed", "high"
 
    signals = bad + [f"[OK] {s}" for s in good]
    if len(bad) == 1 and not is_legacy:
        signals.insert(0, "⚠️ Chỉ 1 tín hiệu cần OCR — nên kiểm tra thủ công.")
 
    return ClassifyResult(
        file=pdf_path, classification=cls, confidence=conf,
        total_pages=total_pages, pages_analyzed=n,
        avg_chars_per_page=avg_chars, text_page_ratio=text_ratio,
        has_embedded_fonts=fonts_ok, image_dom_pages=img_dom,
        garbage_char_ratio=garbage_ratio, signals=signals,
    )


In [7]:
# ╔══════════════════════════════════════════════════════════╗
# ║          CELL 7 — Đọc PDF (hỗ trợ layout 2 cột)        ║
# ╚══════════════════════════════════════════════════════════╝

def _detect_two_columns(page, min_gap: int = 40) -> Optional[float]:
    """
    Phát hiện trang có layout 2 cột bằng cách phân tích
    phân bố tọa độ X của các từ trên trang.

    Ý tưởng:
      - Chia trang thành 20 dải dọc (bins) theo trục X
      - Đếm số từ rơi vào mỗi dải
      - Nếu có "vùng trống" (bin gần như rỗng) ở giữa trang
        → đó là khoảng cách giữa 2 cột
      - Trả về tọa độ X của đường chia cột, hoặc None nếu là 1 cột

    Args:
        page    : pdfplumber page object
        min_gap : Số bin liên tiếp gần rỗng để coi là khoảng cách cột

    Returns:
        float  — tọa độ X đường chia cột (nếu phát hiện 2 cột)
        None   — nếu là layout 1 cột
    """
    words = page.extract_words()
    if len(words) < 20:        # quá ít từ → không đủ dữ liệu
        return None

    page_w = page.width
    n_bins = 20
    bin_w  = page_w / n_bins

    # Đếm số từ theo từng bin dọc
    counts = [0] * n_bins
    for w in words:
        cx  = (w["x0"] + w["x1"]) / 2     # tâm X của từ
        idx = min(int(cx / bin_w), n_bins - 1)
        counts[idx] += 1

    # Tìm vùng trống ở giữa trang (bỏ qua 25% đầu và 25% cuối)
    lo = n_bins // 4        # bin 5
    hi = 3 * n_bins // 4    # bin 15
    max_count = max(counts) or 1
    threshold = max_count * 0.1   # bin có < 10% max → coi là trống

    # Tìm chuỗi bin trống dài nhất trong vùng giữa
    best_start, best_len = -1, 0
    cur_start,  cur_len  = -1, 0

    for i in range(lo, hi):
        if counts[i] <= threshold:
            if cur_len == 0:
                cur_start = i
            cur_len += 1
            if cur_len > best_len:
                best_len  = cur_len
                best_start = cur_start
        else:
            cur_len = 0

    if best_len < 1:           # không tìm thấy vùng trống
        return None

    # Tọa độ X của đường chia = giữa vùng trống
    gap_center = (best_start + best_len / 2) * bin_w

    # Kiểm tra thêm: cả 2 cột phải có nội dung
    left_words  = sum(1 for w in words if w["x1"] < gap_center)
    right_words = sum(1 for w in words if w["x0"] > gap_center)
    if left_words < 5 or right_words < 5:
        return None

    return gap_center


def _extract_two_column_text(page, split_x: float, cfg) -> tuple[str, list]:
    """
    Trích text từ trang 2 cột bằng cách chia bbox trái/phải.

    Quy trình:
      1. Crop trang thành cột trái (x: 0 → split_x)
      2. Crop trang thành cột phải (x: split_x → page_w)
      3. Extract text từng cột riêng
      4. Ghép: hết cột trái → sang cột phải

    Returns:
        (text, tables) — text đã ghép đúng thứ tự, bảng từ cả 2 cột
    """
    page_h = page.height
    page_w = page.width

    # Crop từng cột
    left_page  = page.crop((0,       0, split_x, page_h))
    right_page = page.crop((split_x, 0, page_w,  page_h))

    left_text  = (left_page.extract_text(
                    layout=cfg.preserve_layout) or "").strip()
    right_text = (right_page.extract_text(
                    layout=cfg.preserve_layout) or "").strip()

    # Ghép cột trái trước, cột phải sau
    text = left_text
    if right_text:
        text = text + "\n\n" + right_text if text else right_text

    # Trích bảng từ cả 2 cột (nếu bật)
    tables = []
    if cfg.extract_tables:
        for col_page in (left_page, right_page):
            for t in (col_page.extract_tables() or []):
                if t:
                    tables.append(t)

    return text, tables


def _read_direct(pdf_path: str, cfg: Config) -> list:
    """
    Trích text + bảng trực tiếp bằng pdfplumber.
    Tự động phát hiện và xử lý layout 1 cột / 2 cột mỗi trang.

    Layout detection logic:
      - Mỗi trang được phân tích độc lập
      - Trang 2 cột → crop + extract từng cột, ghép đúng thứ tự
      - Trang 1 cột → extract bình thường như cũ
    """
    pages = []
    col2_count = 0    # đếm số trang 2 cột để log

    with pdfplumber.open(pdf_path) as pdf:
        limit = min(len(pdf.pages), cfg.max_pages) if cfg.max_pages else len(pdf.pages)

        for i in range(limit):
            page = pdf.pages[i]

            # ── Phát hiện layout ───────────────────────────
            split_x = _detect_two_columns(page)
            is_two_col = split_x is not None

            if is_two_col:
                col2_count += 1
                text, tables = _extract_two_column_text(page, split_x, cfg)
                method_label = "direct-2col"
            else:
                text   = (page.extract_text(
                            layout=cfg.preserve_layout) or "").strip()
                tables = [t for t in (page.extract_tables() or []) if t] \
                         if cfg.extract_tables else []
                method_label = "direct-1col"

            pages.append(PageContent(
                page_number = i + 1,
                text        = text,
                tables      = tables,
                method      = method_label,
                char_count  = len(text),
            ))

            print(
                f"  [{method_label}] Trang {i+1}/{limit}  "
                f"{len(text):,} ký tự  {len(tables)} bảng"
                + (f"  [split_x={split_x:.0f}px]" if is_two_col else ""),
                end="\r"
            )

    print()
    if col2_count:
        print(f"  📐 Phát hiện {col2_count}/{limit} trang layout 2 cột")

    # ── Kiểm tra encoding cũ (giữ nguyên từ code gốc) ─────
    sample = " ".join(p.text for p in pages[:3])
    if _is_legacy_vietnamese_font(sample):
        raise ValueError(
            "Text trích ra bị lỗi encoding (TCVN3/VNI/ABC). "
            "Chuyển sang OCR để đọc chính xác."
        )

    return pages


def _read_ocr(pdf_path: str, cfg: Config) -> list:
    """Rasterize từng trang rồi OCR bằng pytesseract."""
    # OCR đọc theo ảnh nên KHÔNG bị ảnh hưởng bởi layout 2 cột
    # → giữ nguyên hoàn toàn, không cần sửa
    print(f"  🔄 Rasterize PDF ở {cfg.ocr_dpi} DPI...")
    images = convert_from_path(pdf_path, dpi=cfg.ocr_dpi, last_page=cfg.max_pages)
    pages  = []
    for i, img in enumerate(images):
        print(f"  [OCR] Trang {i+1}/{len(images)}...", end="\r")
        text = pytesseract.image_to_string(
            img, lang=cfg.ocr_language, config="--oem 3 --psm 3"
        ).strip()
        pages.append(PageContent(
            page_number = i + 1,
            text        = text,
            tables      = [],
            method      = "ocr",
            char_count  = len(text),
        ))
    print()
    return pages


def read_pdf(pdf_path: str, cfg: Config, force: Optional[str] = None) -> ReadResult:
    """
    Đọc một file PDF — tự phân loại rồi chọn đúng phương pháp.
    Giữ nguyên hoàn toàn từ code gốc.

    Args:
        pdf_path : Đường dẫn file PDF.
        cfg      : Config dùng chung cho phân loại lẫn đọc.
        force    : \"direct\" hoặc \"ocr\" để ép buộc phương pháp.
                   None = tự động theo kết quả phân loại.
    """
    t0 = time.time()

    print(f"\n📄 {os.path.basename(pdf_path)}")
    cls = classify_pdf(pdf_path, cfg)
    print(cls)

    if cls.error:
        return ReadResult(
            file=pdf_path, classification="unknown", confidence="low",
            total_pages=0, pages_read=0, method="none", pages=[],
            elapsed_seconds=time.time() - t0, error=cls.error,
        )

    method = force if force else (
        "direct" if cls.classification == "No OCR needed" else "ocr"
    )
    print(f"⚙️  Phương pháp: {method.upper()}")

    try:
        pages = _read_direct(pdf_path, cfg) if method == "direct" \
                else _read_ocr(pdf_path, cfg)
    except Exception as e:
        if method == "direct":
            print(f"\n  ⚠️  Direct thất bại ({e}), chuyển sang OCR...")
            try:
                pages  = _read_ocr(pdf_path, cfg)
                method = "ocr"
            except Exception as e2:
                return ReadResult(
                    file=pdf_path, classification=cls.classification,
                    confidence=cls.confidence, total_pages=cls.total_pages,
                    pages_read=0, method="ocr", pages=[],
                    elapsed_seconds=time.time() - t0, error=str(e2),
                )
        else:
            return ReadResult(
                file=pdf_path, classification=cls.classification,
                confidence=cls.confidence, total_pages=cls.total_pages,
                pages_read=0, method=method, pages=[],
                elapsed_seconds=time.time() - t0, error=str(e),
            )

    return ReadResult(
        file=pdf_path, classification=cls.classification,
        confidence=cls.confidence, total_pages=cls.total_pages,
        pages_read=len(pages), method=method, pages=pages,
        elapsed_seconds=time.time() - t0,
    )
def get_pdf_text_smart(pdf_path: str, cfg: Config) -> str:
    """
    Tự động chọn phương pháp đọc PDF phù hợp:
      - No OCR needed  → read_pdf_text_pymupdf() (nhanh, giữ layout 2 cột)
      - OCR required   → _read_ocr() qua pytesseract, ghép text các trang

    Đây là hàm bridge nối hệ thống classify/OCR (Cell 5–8)
    với pipeline tóm tắt TextRank (Cell 17).
    """
    cls = classify_pdf(pdf_path, cfg)
    print(cls)  # in kết quả phân loại để debug

    if cls.error:
        raise RuntimeError(f"Không đọc được PDF: {cls.error}")

    if cls.classification == "No OCR needed":
        print("⚡ Dùng PyMuPDF direct (không cần OCR)")
        return read_pdf_text_pymupdf(pdf_path, max_pages=cfg.max_pages)
    else:
        print(f"🔎 Dùng OCR (lý do: confidence={cls.confidence})")
        pages = _read_ocr(pdf_path, cfg)
        return "\n\n".join(p.text for p in pages if p.text)

In [8]:
# ╔══════════════════════════════════════════════════════════╗
# ║          CELL 8 — Chạy                                  ║
# ╚══════════════════════════════════════════════════════════╝

cfg = Config(
    ocr_language    = "vie+eng",
    ocr_dpi         = 300,
    extract_tables  = True,
    preserve_layout = True,
    max_pages       = None,
)

PDF_FILES = [
    "/kaggle/input/datasets/vietnguyencong123/dataops2/1.pdf",
    # thêm file khác vào đây nếu cần
]

results = []
for path in PDF_FILES:
    result = read_pdf(path, cfg)
    print(result)

    # Lưu text ra file để dùng lại nếu kernel restart
    base = os.path.splitext(os.path.basename(path))[0]
    result.save_text(f"/kaggle/working/{base}_extracted.txt")

    results.append(result)

# ── Preview trang đầu tiên ──
for r in results:
    if r.pages:
        print(f"\n{'─'*55}")
        print(f"PREVIEW — {os.path.basename(r.file)}")
        print(f"{'─'*55}")
        print(r.pages[0].text[:1000])

print(f"\n✅ Xong! results có {len(results)} file")


📄 1.pdf
───────────────────────────────────────────────────────
  File       : 1.pdf
  Kết quả    : 🔴 OCR required
  Độ tin cậy : high
───────────────────────────────────────────────────────
  Tổng trang          : 13 (phân tích: 13)
  Ký tự TB/trang      : 3424.4
  Tỷ lệ trang có text : 100.0%
  Font nhúng          : Có
  Trang bị ảnh phủ    : 0
  Tỷ lệ ký tự rác     : 2.5%
───────────────────────────────────────────────────────
  • Phát hiện font encoding cũ (TCVN3/VNI/ABC) → text bị sai ký tự, cần OCR.
  • [OK] Ký tự TB/trang đủ lớn (3424).
  • [OK] 100% trang có văn bản.
  • [OK] Font nhúng đầy đủ.
  • [OK] Ký tự rác thấp (2.5%).
───────────────────────────────────────────────────────
⚙️  Phương pháp: OCR
  🔄 Rasterize PDF ở 300 DPI...
  [OCR] Trang 13/13...
═══════════════════════════════════════════════════════
  File         : 1.pdf
  Phân loại    : OCR required (high)
  Phương pháp  : OCR
  Trang đọc    : 13/13
  Tổng ký tự   : 43,205
  Bảng tìm được: 0
  Thời gian    : 81.7s


In [9]:
# ╔══════════════════════════════════════════════════════════╗
# ║     CELL 8.5 — Cài unsloth cho inference                ║
# ╚══════════════════════════════════════════════════════════╝
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes -q

# Kiểm tra
from unsloth import FastLanguageModel
import torch
print("✅ Unsloth ready!")
print(f"   GPU: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 103.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 90.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 105.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [10]:
# ╔══════════════════════════════════════════════════════════╗
# ║     CELL 9 — Load base model + gắn LoRA adapter         ║
# ╚══════════════════════════════════════════════════════════╝
import glob, os, torch
from unsloth import FastLanguageModel
from peft import PeftModel

# ── Tìm đường dẫn adapter ──────────────────────────────────
_search = glob.glob("/kaggle/input/datasets/vietnguyencong123/dataops2/qwen2.5_textrank_lora-20260510T081852Z-3-001/qwen2.5_textrank_lora", recursive=True)
ADAPTER_PATH = _search[0] if _search else "/kaggle/input/dataops2/qwen2.5_textrank_lora"
print(f"✅ Adapter path : {ADAPTER_PATH}")
print(f"   Files found  : {os.listdir(ADAPTER_PATH)}")

# ── Bước 1: Load base model (giống hệt lúc fine-tune) ──────
print("\n🔄 Loading base model Qwen2.5-1.5B...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length= 4096,
    dtype         = None,
    load_in_4bit  = True,
)

# ── Bước 2: Gắn LoRA adapter đã fine-tune vào ──────────────
print("🔄 Gắn LoRA adapter...")
model = PeftModel.from_pretrained(model, ADAPTER_PATH)

# ── Bước 3: Bật chế độ inference nhanh ─────────────────────
FastLanguageModel.for_inference(model)

print("\n✅ Model sẵn sàng!")
print(f"   GPU  : {torch.cuda.get_device_name(0)}")
print(f"   VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

✅ Adapter path : /kaggle/input/datasets/vietnguyencong123/dataops2/qwen2.5_textrank_lora-20260510T081852Z-3-001/qwen2.5_textrank_lora
   Files found  : ['adapter_model.safetensors', 'adapter_config.json', 'README.md', 'tokenizer.json', 'tokenizer_config.json', 'chat_template.jinja']

🔄 Loading base model Qwen2.5-1.5B...
==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
🔄 Gắn LoRA adapter...


/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(



✅ Model sẵn sàng!
   GPU  : Tesla T4
   VRAM : 15.6 GB


In [11]:
# ╔══════════════════════════════════════════════════════════╗
# ║     CELL 10 — Chunking text đều nhau                    ║
# ╚══════════════════════════════════════════════════════════╝
from typing import Optional
 
def chunk_text(
    text: str,
    chunk_size: int   = 300,    # số từ mỗi chunk
    overlap:    int   = 30,     # số từ chồng lấp giữa các chunk (giữ ngữ cảnh)
) -> list[dict]:
    """
    Chia text thành các chunk đều nhau theo số từ.
 
    Args:
        text       : Toàn bộ văn bản.
        chunk_size : Số từ mỗi chunk (mặc định 300 từ ≈ ~1 đoạn văn bản).
        overlap    : Số từ chồng lấp đầu/cuối để không mất ngữ cảnh.
 
    Returns:
        list[dict] với keys: chunk_id, text, word_count, start_word, end_word
    """
    words  = text.split()
    total  = len(words)
    chunks = []
    step   = chunk_size - overlap
    i      = 0
 
    while i < total:
        end        = min(i + chunk_size, total)
        chunk_words= words[i:end]
        chunks.append({
            "chunk_id"  : len(chunks),
            "text"      : " ".join(chunk_words),
            "word_count": len(chunk_words),
            "start_word": i,
            "end_word"  : end,
        })
        if end >= total:
            break
        i += step
 
    print(f"📦 Tổng {len(chunks)} chunk  |  "
          f"~{chunk_size} từ/chunk  |  overlap {overlap} từ  |  "
          f"Tổng {total:,} từ")
    return chunks

In [12]:
# ╔══════════════════════════════════════════════════════════╗
# ║     CELL 11 — Dùng model chấm điểm từng chunk           ║
# ╚══════════════════════════════════════════════════════════╝
import torch
 
# Template giống hệt lúc fine-tune
PROMPT_TEMPLATE = """<|im_start|>system
Bạn là một chuyên gia phân tích dữ liệu. Nhiệm vụ của bạn là đọc bài báo khoa học và trích xuất các đoạn văn bản cốt lõi nhất theo yêu cầu.<|im_end|>
<|im_start|>user
{instruction}
 
Nội dung bài báo:
{input}<|im_end|>
<|im_start|>assistant
"""
 
INSTRUCTION = (
    "Đọc đoạn văn bản sau và đánh giá mức độ quan trọng của nó trong bài báo khoa học. "
    "Trả lời theo định dạng:\n"
    "ĐIỂM: <số từ 1 đến 10>\n"
    "LÝ DO: <1 câu giải thích ngắn gọn>"
)
 
 
def score_chunk(chunk_text: str, max_new_tokens: int = 80) -> dict:
    """
    Dùng model fine-tune để chấm điểm quan trọng (1–10) cho 1 chunk.
    Trả về dict: {score: float, reason: str, raw: str}
    """
    prompt = PROMPT_TEMPLATE.format(
        instruction=INSTRUCTION,
        input=chunk_text,
    )
 
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
 
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens   = max_new_tokens,
            do_sample        = False,       # greedy — ổn định hơn cho scoring
            temperature      = 1.0,
            pad_token_id     = tokenizer.eos_token_id,
        )
 
    # Chỉ lấy phần model sinh ra (bỏ phần prompt)
    new_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    raw     = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
 
    # Parse điểm
    score  = _parse_score(raw)
    reason = _parse_reason(raw)
    return {"score": score, "reason": reason, "raw": raw}
 
 
def _parse_score(text: str) -> float:
    """Trích số điểm từ output của model."""
    import re
    # Tìm pattern "ĐIỂM: X" hoặc "điểm: X" hoặc chỉ số đứng đầu
    m = re.search(r"(?:ĐIỂM|điểm|Score|score)\s*[:\-]\s*(\d+(?:\.\d+)?)", text)
    if m:
        return min(float(m.group(1)), 10.0)
    # Fallback: lấy số đầu tiên tìm được
    nums = re.findall(r"\b(\d+(?:\.\d+)?)\b", text)
    if nums:
        val = float(nums[0])
        if 1 <= val <= 10:
            return val
    return 5.0   # điểm mặc định nếu không parse được
 
 
def _parse_reason(text: str) -> str:
    """Trích lý do từ output của model."""
    import re
    m = re.search(r"(?:LÝ DO|lý do|Reason|reason)\s*[:\-]\s*(.+)", text, re.IGNORECASE)
    if m:
        return m.group(1).strip()
    # Fallback: dòng thứ 2 trở đi
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    return lines[1] if len(lines) > 1 else text[:120]

In [13]:
# ╔══════════════════════════════════════════════════════════╗
# ║     CELL 12 — Pipeline chính: chunk → score → top-k     ║
# ╚══════════════════════════════════════════════════════════╝
 
def select_top_k_chunks(
    text:        str,
    k:           int  = 5,       # số chunk quan trọng cần chọn
    chunk_size:  int  = 300,     # số từ mỗi chunk
    overlap:     int  = 30,      # số từ chồng lấp
    verbose:     bool = True,
) -> list[dict]:
    """
    Pipeline đầy đủ:
      1. Chunk text đều nhau
      2. Dùng model fine-tune chấm điểm từng chunk
      3. Trả về top-k chunk quan trọng nhất (đã sort theo điểm)
 
    Returns:
        list[dict] gồm k chunk, mỗi chunk có thêm: score, reason
    """
    # Bước 1 — Chunk
    chunks = chunk_text(text, chunk_size=chunk_size, overlap=overlap)
 
    if len(chunks) <= k:
        print(f"⚠️  Chỉ có {len(chunks)} chunk, trả về tất cả.")
        k = len(chunks)
 
    # Bước 2 — Chấm điểm từng chunk
    print(f"\n🤖 Đang chấm điểm {len(chunks)} chunk bằng model fine-tune...")
    for i, chunk in enumerate(chunks):
        result = score_chunk(chunk["text"])
        chunk["score"]  = result["score"]
        chunk["reason"] = result["reason"]
        chunk["raw"]    = result["raw"]
        if verbose:
            print(f"  Chunk {i+1:>3}/{len(chunks)}  "
                  f"[{chunk['start_word']:>5}–{chunk['end_word']:>5} từ]  "
                  f"Điểm: {chunk['score']:.1f}  {chunk['reason'][:60]}")
 
    # Bước 3 — Chọn top-k (giữ thứ tự xuất hiện trong văn bản)
    sorted_chunks = sorted(chunks, key=lambda c: c["score"], reverse=True)
    top_k         = sorted(sorted_chunks[:k], key=lambda c: c["chunk_id"])
 
    print(f"\n✅ Đã chọn top {k} chunk quan trọng nhất:")
    print("─" * 60)
    for rank, c in enumerate(top_k, 1):
        print(f"  #{rank}  Chunk {c['chunk_id']+1}  "
              f"(từ {c['start_word']}–{c['end_word']})  "
              f"Điểm: {c['score']:.1f}")
        print(f"       {c['text'][:120]}...")
        print()
 
    return top_k

In [14]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 14 — Load Qwen2.5-7B để biên tập TextRank         ║
# ╚══════════════════════════════════════════════════════════╝

import torch
from unsloth import FastLanguageModel

SUM_MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct"
SUM_MAX_SEQ_LENGTH = 8192

print(f"Đang load {SUM_MODEL_NAME}...")

sum_model, sum_tokenizer = FastLanguageModel.from_pretrained(
    model_name     = SUM_MODEL_NAME,
    max_seq_length = SUM_MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,
)
FastLanguageModel.for_inference(sum_model)

if sum_tokenizer.pad_token_id is None:
    sum_tokenizer.pad_token = sum_tokenizer.eos_token

allocated = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9

print("Qwen2.5-7B polish model đã sẵn sàng.")
print(f"GPU          : {torch.cuda.get_device_name(0)}")
print(f"VRAM đã dùng : {allocated:.1f} GB / {total:.1f} GB")
print(f"Context      : {SUM_MAX_SEQ_LENGTH} tokens")

Đang load unsloth/Qwen2.5-7B-Instruct...
==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Qwen2.5-7B polish model đã sẵn sàng.
GPU          : Tesla T4
VRAM đã dùng : 8.8 GB / 15.6 GB
Context      : 8192 tokens


In [15]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 15 — Tính K từ số từ mục tiêu                    ║
# ╚══════════════════════════════════════════════════════════╝
 
def compute_k_from_target_words(
    target_words:       int,
    chunk_size:         int   = 300,
    overlap:            int   = 30,
    compression_ratio:  float = 0.75,
    min_k:              int   = 1,
    max_k:              int   = 50,
) -> int:
    """
    Tính ngược số K chunk cần lấy để sau khi LLM nén lại
    ra được văn bản ~target_words từ.
 
    Công thức:
        effective_per_chunk = (chunk_size - overlap) × compression_ratio
        K = round(target_words / effective_per_chunk)
 
    Args:
        target_words       : Số từ mong muốn trong bản tóm tắt cuối.
        chunk_size         : Số từ/chunk — phải khớp với chunk_text().
        overlap            : Từ chồng lấp — phải khớp với chunk_text().
        compression_ratio  : Tỷ lệ nén LLM (0–1). 0.75 = giữ 75% nội dung.
        min_k / max_k      : Giới hạn an toàn.
    """
    effective = (chunk_size - overlap) * compression_ratio
    if effective <= 0:
        raise ValueError("chunk_size phải lớn hơn overlap.")
 
    k = max(min_k, min(round(target_words / effective), max_k))
 
    print("─" * 50)
    print(f"  Mục tiêu       : {target_words:,} từ")
    print(f"  Chunk / overlap: {chunk_size} / {overlap} từ")
    print(f"  Tỷ lệ nén      : {compression_ratio*100:.0f}%")
    print(f"  Từ hiệu dụng   : {effective:.1f} từ/chunk")
    print(f"  K              : {round(target_words/effective):.2f} → K = {k}")
    print("─" * 50)
    return k

In [22]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 16 — TextRank summarizer (Tổng quát hóa đa miền)  ║
# ╚══════════════════════════════════════════════════════════╝

import os
import re
import math
import time
import html
import unicodedata
from dataclasses import dataclass
from collections import Counter, defaultdict
from typing import Optional

import fitz
import numpy as np
import torch

try:
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()
except Exception:
    pass


# Stopwords tiếng Việt chung
VI_STOPWORDS = {
    "và", "là", "của", "có", "cho", "trong", "được", "với", "các", "những",
    "một", "này", "đó", "khi", "từ", "đến", "trên", "dưới", "về", "theo",
    "tại", "bởi", "do", "nên", "đã", "đang", "sẽ", "rằng", "thì", "mà",
    "như", "hoặc", "nếu", "để", "ra", "vào", "sau", "trước", "giữa",
    "bị", "bằng", "không", "cũng", "nhiều", "ít", "hơn", "nhất", "gồm",
    "qua", "lại", "năm", "ngày", "tháng", "bài", "báo", "nghiên", "cứu",
    "tỷ", "lệ", "kết", "quả",
}

# Từ khóa quan trọng đa lĩnh vực (Khoa học, Lịch sử, Kinh tế, Kỹ thuật)
IMPORTANT_TERMS = {
    "mục tiêu", "phương pháp", "kết quả", "kết luận", "tóm tắt", "tổng quan",
    "tóm lại", "nhìn chung", "giai đoạn", "chính sách", "tổ chức",
    "quản lý", "biến động", "lãnh thổ", "phát triển", "thực thi", "chủ quyền",
    "ảnh hưởng", "vai trò", "ý nghĩa", "đánh giá", "tác động", "đặc điểm",
    "đối tượng", "quá trình", "phân tích"
}

SECTION_HEADINGS = [
    "ĐẶT VẤN ĐỀ",
    "MỞ ĐẦU",
    "GIỚI THIỆU",
    "ĐỐI TƯỢNG VÀ PHƯƠNG PHÁP",
    "PHƯƠNG PHÁP NGHIÊN CỨU",
    "KẾT QUẢ",
    "BÀN LUẬN",
    "KẾT LUẬN",
    "TỔNG KẾT",
]


def _nfc(text: str) -> str:
    return unicodedata.normalize("NFC", text or "")

def _strip_accents(text: str) -> str:
    text = unicodedata.normalize("NFD", text)
    text = "".join(ch for ch in text if unicodedata.category(ch) != "Mn")
    return text.replace("đ", "d").replace("Đ", "D")

def _squeeze(text: str) -> str:
    text = _nfc(text or "")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\s+([,.;:!?%)])", r"\1", text)
    text = re.sub(r"([(])\s+", r"\1", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def word_count(text: str) -> int:
    return len((text or "").split())

def _words_to_lines(words: list, y_tol: float = 3.0) -> str:
    if not words:
        return ""
    words = sorted(words, key=lambda w: (w[1], w[0]))
    lines = []
    cur = []
    cur_y = None
    for w in words:
        x0, y0, x1, y1, token = w[:5]
        if cur_y is None or abs(y0 - cur_y) <= y_tol:
            cur.append(w)
            cur_y = y0 if cur_y is None else (cur_y * 0.8 + y0 * 0.2)
        else:
            cur = sorted(cur, key=lambda z: z[0])
            lines.append(" ".join(str(z[4]) for z in cur))
            cur = [w]
            cur_y = y0
    if cur:
        cur = sorted(cur, key=lambda z: z[0])
        lines.append(" ".join(str(z[4]) for z in cur))
    return "\n".join(lines)

def _detect_two_column_split(words: list, page_width: float) -> Optional[float]:
    if len(words) < 80:
        return None
    centers = [float((w[0] + w[2]) / 2) for w in words]
    n_bins = 32
    counts = [0] * n_bins
    for x in centers:
        idx = min(n_bins - 1, max(0, int(x / page_width * n_bins)))
        counts[idx] += 1
    lo = int(n_bins * 0.35)
    hi = int(n_bins * 0.65)
    max_count = max(counts) or 1
    threshold = max(2, max_count * 0.18)

    best_start, best_len = -1, 0
    cur_start, cur_len = -1, 0
    for i in range(lo, hi):
        if counts[i] <= threshold:
            if cur_len == 0:
                cur_start = i
            cur_len += 1
            if cur_len > best_len:
                best_start, best_len = cur_start, cur_len
        else:
            cur_len = 0
    if best_len < 1:
        return None
    split_x = (best_start + best_len / 2) / n_bins * page_width
    left = sum(1 for x in centers if x < split_x)
    right = sum(1 for x in centers if x >= split_x)
    if left < 30 or right < 30:
        return None
    if min(left, right) / max(left, right) < 0.25:
        return None
    return split_x

def _page_text_by_columns(page) -> str:
    words = page.get_text("words")
    if not words:
        return page.get_text("text", sort=True) or ""
    page_w = float(page.rect.width)
    page_h = float(page.rect.height)
    words = [
        w for w in words
        if page_h * 0.035 <= float(w[1]) <= page_h * 0.965
    ]
    split_x = _detect_two_column_split(words, page_w)
    if split_x is None:
        return _words_to_lines(words)
    left = [w for w in words if (float(w[0]) + float(w[2])) / 2 < split_x]
    right = [w for w in words if (float(w[0]) + float(w[2])) / 2 >= split_x]
    return _words_to_lines(left) + "\n\n" + _words_to_lines(right)

def read_pdf_text_pymupdf(pdf_path: str, max_pages: Optional[int] = None) -> str:
    pages = []
    with fitz.open(pdf_path) as doc:
        limit = min(doc.page_count, max_pages) if max_pages else doc.page_count
        for i in range(limit):
            text = _page_text_by_columns(doc.load_page(i))
            if text and text.strip():
                pages.append(text)
    return "\n\n".join(pages)

def extract_title(raw_text: str) -> str:
    text = _nfc(raw_text)
    lines = []
    for line in text.splitlines():
        line = _squeeze(line)
        if len(line.split()) >= 8 and not re.search(r"doi|tạp chí|bản quyền", line, re.I):
            lines.append(line)
    return lines[0] if lines else "Tài liệu khoa học/Nghiên cứu"

def remove_abstract_blocks(text: str) -> str:
    text = re.sub(
        r"(?is)\bTÓM TẮT\b.*?\bTừ kh[oó]a\b.*?(?=\bĐẶT VẤN ĐỀ\b|\bI\.\s*ĐẶT VẤN ĐỀ\b|\b1\.\s*ĐẶT VẤN ĐỀ\b|\bGIỚI THIỆU\b)",
        " ",
        text,
    )
    text = re.sub(
        r"(?is)\bABSTRACT\s*:?.*?\bKeywords?\b.*?(?=\bTÓM TẮT\b|\bĐẶT VẤN ĐỀ\b|\bI\.\s*ĐẶT VẤN ĐỀ\b|\bGIỚI THIỆU\b)",
        " ",
        text,
    )
    return text

def clean_pdf_body_text(raw_text: str) -> str:
    text = _nfc(raw_text).replace("\r", "\n")
    text = remove_abstract_blocks(text)
    text = re.split(r"(?im)^\s*(TÀI LIỆU THAM KHẢO|REFERENCES)\s*$", text)[0]

    body_start = None
    for pat in [
        r"(?im)^\s*(I\.|1\.)?\s*ĐẶT VẤN ĐỀ\s*$",
        r"(?im)^\s*(I\.|1\.)?\s*MỞ ĐẦU\s*$",
        r"(?im)^\s*(I\.|1\.)?\s*GIỚI THIỆU\s*$",
    ]:
        m = re.search(pat, text)
        if m:
            body_start = m.start()
            break
    if body_start is not None:
        text = text[body_start:]

    # Noise patterns khái quát cho cả Y học, KHXH, Kỹ thuật
    noise_patterns = [
        r"^Tạp chí Khoa học", r"^Tập\s+\d+", r"^Bản quyền", r"^DOI\s*:",
        r"^https?://", r"^\*?Tác giả liên hệ", r"^Điện thoại\s*:", r"^Email\s*:",
        r"^Thông tin bài đăng", r"^Ngày nhận bài\s*:", r"^Ngày phản biện\s*:",
        r"^Ngày duyệt bài\s*:", r"^PGS\.?TS", r"^TS\.", r"^ThS\.", r"^GS\.", r"^BS\.",
        r"^Viện\s+", r"^Trường Đại học",
    ]

    kept = []
    for line in text.splitlines():
        line = _squeeze(line)
        if not line:
            kept.append("")
            continue
        if re.fullmatch(r"\d{1,3}", line):
            continue
        if any(re.search(p, line, flags=re.IGNORECASE) for p in noise_patterns):
            continue
        
        letters = len(re.findall(r"[A-Za-zÀ-ỹĐđ]", line))
        digits = len(re.findall(r"\d", line))
        if len(line.split()) <= 4 and digits > letters:
            continue
        kept.append(line)

    text = "\n".join(kept)
    text = re.sub(r"(?<=\w)-\n(?=\w)", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return _squeeze(text)

def protect_abbreviations(text: str) -> str:
    protected = {
        "cs.": "cs<dot>", "ThS.": "ThS<dot>", "TS.": "TS<dot>",
        "BS.": "BS<dot>", "PGS.": "PGS<dot>", "GS.": "GS<dot>",
        "vs.": "vs<dot>", "v.v.": "vv<dot>"
    }
    for k, v in protected.items():
        text = text.replace(k, v)
    return text

def unprotect_abbreviations(text: str) -> str:
    return text.replace("<dot>", ".")

def split_sentences(text: str) -> list[str]:
    text = protect_abbreviations(_squeeze(text))
    for h in SECTION_HEADINGS:
        text = re.sub(rf"\b{re.escape(h)}\b", f". {h}. ", text, flags=re.IGNORECASE)

    raw_sents = re.split(r"(?<=[.!?])\s+(?=[A-ZÀ-ỸĐ0-9])", text)
    sentences = []

    for s in raw_sents:
        s = unprotect_abbreviations(_squeeze(s))
        if not s:
            continue
        wc = word_count(s)
        if wc < 7 or wc > 90:
            continue
        if re.search(r"(?i)\b(tạp chí|doi|bản quyền|email|điện thoại)\b", s):
            continue
        if re.search(r"\bKIẾN NGHỊ\b", s, flags=re.IGNORECASE):
            s = re.split(r"\bKIẾN NGHỊ\b", s, flags=re.IGNORECASE)[0].strip()
            if word_count(s) < 7:
                continue
                
        # Regex linh hoạt để bỏ qua các câu chú thích bảng biểu
        table_markers = [
            r"nguyên nhân\s*\( ?% ?\)", r"\bBảng\s+\d+", r"\bBiểu đồ\s+\d+", r"\bHình\s+\d+", r"\(N\s*="
        ]
        if sum(1 for p in table_markers if re.search(p, s, flags=re.IGNORECASE)) >= 2:
            continue
            
        alpha = len(re.findall(r"[A-Za-zÀ-ỹĐđ]", s))
        if alpha < 20:
            continue
        sentences.append(s)

    out, seen = [], set()
    for s in sentences:
        key = _strip_accents(s.lower())
        key = re.sub(r"[^a-z0-9à-ỹđ]+", " ", key)
        key = " ".join(key.split()[:18])
        if key in seen:
            continue
        seen.add(key)
        out.append(s)
    return out

def tokenize_for_rank(sentence: str) -> list[str]:
    s = _strip_accents(sentence.lower())
    words = re.findall(r"[a-zA-ZÀ-ỹĐđ]{2,}", s)
    return [w for w in words if w not in VI_STOPWORDS and len(w) >= 2]

def build_tfidf_matrix(sentences: list[str], max_terms: int = 1200) -> np.ndarray:
    tokenized = [tokenize_for_rank(s) for s in sentences]
    df = Counter()
    tf_list = []
    for toks in tokenized:
        tf = Counter(toks)
        tf_list.append(tf)
        df.update(tf.keys())

    terms = [t for t, c in df.most_common(max_terms) if c >= 2 or len(sentences) < 30]
    vocab = {t: i for i, t in enumerate(terms)}
    n = len(sentences)
    mat = np.zeros((n, len(vocab)), dtype=np.float32)

    for i, tf in enumerate(tf_list):
        for term, count in tf.items():
            j = vocab.get(term)
            if j is None:
                continue
            idf = math.log((1 + n) / (1 + df[term])) + 1.0
            mat[i, j] = (1.0 + math.log(count)) * idf

    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return mat / norms

def pagerank_scores(similarity: np.ndarray, damping: float = 0.85, max_iter: int = 100, tol: float = 1e-6) -> np.ndarray:
    n = similarity.shape[0]
    if n == 0:
        return np.array([])
    W = similarity.copy()
    np.fill_diagonal(W, 0.0)
    W[W < 0.05] = 0.0
    row_sums = W.sum(axis=1, keepdims=True)
    W = np.divide(W, row_sums, out=np.zeros_like(W), where=row_sums != 0)
    scores = np.ones(n, dtype=np.float32) / n
    base = (1.0 - damping) / n
    for _ in range(max_iter):
        new_scores = base + damping * (W.T @ scores)
        if np.linalg.norm(new_scores - scores, ord=1) < tol:
            scores = new_scores
            break
        scores = new_scores
    return scores

def section_bonus(sentence: str) -> float:
    s = sentence.lower()
    bonus = 1.0
    if any(term in s for term in IMPORTANT_TERMS):
        bonus += 0.18
    if re.search(r"(?i)\b(mục tiêu|phương pháp|kết quả|kết luận|tóm tắt|tổng quan)\b", s):
        bonus += 0.08
    return bonus

def select_sentences_textrank(
    sentences: list[str],
    base_scores: np.ndarray,
    target_words: int = 500,
    min_words: int = 320,
    redundancy_threshold: float = 0.72,
) -> list[dict]:
    if not sentences:
        return []
    mat = build_tfidf_matrix(sentences)
    similarity = mat @ mat.T
    final_scores = np.array([
        float(base_scores[i]) * section_bonus(sentences[i])
        for i in range(len(sentences))
    ], dtype=np.float32)

    order = list(np.argsort(-final_scores))
    selected = []
    selected_ids = []
    total_words = 0

    def is_redundant(idx: int, threshold: float = redundancy_threshold) -> bool:
        if not selected_ids:
            return False
        return max(float(similarity[idx, j]) for j in selected_ids) > threshold

    def add_idx(idx: int, *, allow_over: bool = False, threshold: float = redundancy_threshold) -> bool:
        nonlocal total_words
        if idx in selected_ids:
            return False
        raw_sent = sentences[idx]
        clean_sent = clean_sentence_for_display(raw_sent)
        if not clean_sent:
            return False
        wc = word_count(clean_sent)
        upper_words = target_words + max(60, int(target_words * 0.15))

        if (not allow_over) and total_words >= min_words and total_words + wc > upper_words:
            return False
        if total_words + wc > upper_words:
            return False
        if is_redundant(idx, threshold=threshold):
            return False
        selected.append({
            "idx": int(idx),
            "score": float(final_scores[idx]),
            "textrank": float(base_scores[idx]),
            "word_count": wc,
            "text": raw_sent,
            "clean_text": clean_sent,
        })
        selected_ids.append(idx)
        total_words += wc
        return True

    def best_idx(pattern: str, pool: Optional[range] = None) -> Optional[int]:
        ids = list(pool) if pool is not None else list(range(len(sentences)))
        candidates = [
            i for i in ids
            if re.search(pattern, sentences[i], flags=re.IGNORECASE)
        ]
        if not candidates:
            return None
        return max(candidates, key=lambda i: final_scores[i])

    n = len(sentences)
    early_pool = range(0, max(1, min(n, int(n * 0.28))))

    # Seed Patterns đa lĩnh vực (Bao quát logic Mở bài -> Quá trình -> Kết luận)
    seed_patterns = [
        (r"mục tiêu|mục đích|nội dung chính|tóm tắt lại|tổng quan|đặt vấn đề", early_pool),
        (r"phương pháp|cách thức|quá trình|giai đoạn|thời kỳ|tiến trình", None),
        (r"tóm lại|nhìn chung|kết luận|kết quả|cho thấy|đánh giá", None),
        (r"số liệu|tỷ lệ|thống kê|thực thi|quản lý|tác động|ý nghĩa", None),
    ]

    for pat, pool in seed_patterns:
        idx = best_idx(pat, pool)
        if idx is not None:
            add_idx(idx, allow_over=True, threshold=0.88)

    for idx in order:
        add_idx(idx)
        if total_words >= min_words:
            break

    if total_words < min_words:
        for idx in order:
            if add_idx(idx, allow_over=False, threshold=0.92):
                if total_words >= min_words:
                    break

    selected = sorted(selected, key=lambda x: x["idx"])
    return selected

def textrank_extract(text: str, target_words: int = 500) -> tuple[list[dict], list[str], np.ndarray]:
    sentences = split_sentences(text)
    if len(sentences) < 5:
        raise RuntimeError(f"Không đủ câu để chạy TextRank: chỉ có {len(sentences)} câu.")
    mat = build_tfidf_matrix(sentences)
    similarity = mat @ mat.T
    scores = pagerank_scores(similarity)
    selected = select_sentences_textrank(
        sentences=sentences,
        base_scores=scores,
        target_words=target_words,
        min_words=max(120, int(target_words * 0.90)),
    )
    return selected, sentences, scores

def extractive_summary_from_selected(selected: list[dict]) -> str:
    cleaned = []
    seen = set()
    for item in selected:
        s = item.get("clean_text") or clean_sentence_for_display(item["text"])
        if not s:
            continue
        key = re.sub(r"\W+", " ", _strip_accents(s.lower())).strip()[:120]
        if key in seen:
            continue
        seen.add(key)
        cleaned.append(s)
    if len(cleaned) <= 4:
        return _squeeze(" ".join(cleaned))
    chunks = []
    step = max(2, math.ceil(len(cleaned) / 4))
    for i in range(0, len(cleaned), step):
        chunks.append(" ".join(cleaned[i:i + step]))
    return _squeeze("\n\n".join(chunks))

def clean_sentence_for_display(sentence: str) -> str:
    """Đã dọn sạch các rule hardcode, chỉ giữ regex làm sạch văn bản chung"""
    s = _squeeze(sentence)
    s = re.split(r"\bKIẾN NGHỊ\b", s, flags=re.IGNORECASE)[0]
    
    # Xóa trích dẫn dạng (1), (23)
    s = re.sub(r"\s*\(\d+\)\s*", " ", s)
    # Xóa trích dẫn dạng [1], [2, 3]
    s = re.sub(r"\s*\[\d+(?:\s*,\s*\d+)*\]\s*", " ", s)
    
    # Xóa các heading bị lẫn vào câu
    s = re.sub(r"\b(BÀN LUẬN|KẾT LUẬN|KẾT QUẢ|PHƯƠNG PHÁP NGHIÊN CỨU)\b", " ", s, flags=re.IGNORECASE)
    
    s = re.sub(r"\s+([,.;:!?%)])", r"\1", s)
    s = re.sub(r"\.{2,}", ".", s)
    s = re.sub(r"\s{2,}", " ", s)
    return _squeeze(s)

def _has_cjk(text: str) -> bool:
    return bool(re.search(
        r"[　-〿㐀-䶿一-鿿豈-﫿＀-￯"
        r"\U00020000-\U0002a6df\U0002a700-\U0002ceaf]",
        text or "",
    ))

_CJK_BAD_WORDS_IDS = None
_CJK_SCANNED = False

def get_cjk_bad_words_ids():
    global _CJK_BAD_WORDS_IDS, _CJK_SCANNED
    if _CJK_SCANNED:
        return _CJK_BAD_WORDS_IDS
    bad_ids = set()
    vocab = sum_tokenizer.get_vocab()
    for token, token_id in vocab.items():
        if _has_cjk(token):
            bad_ids.add(token_id)
    _CJK_SCANNED = True
    _CJK_BAD_WORDS_IDS = [[x] for x in sorted(bad_ids)] if bad_ids else None
    print(f"Đã khóa {len(bad_ids):,} token CJK trong tokenizer.")
    return _CJK_BAD_WORDS_IDS

def strip_bad_output(text: str) -> str:
    text = re.sub(r"<\|im_(start|end)\|>", "", text or "")
    text = re.sub(r"(?im)^\s*(system|user|assistant)\s*:?\s*$", "", text)
    text = re.sub(r"(?im)^\s*#{1,6}\s*", "", text)
    return _squeeze(text)

def qwen_polish_textrank(title: str, selected: list[dict], target_words: int = 500) -> str:
    evidence = "\n".join(
        f"{i+1}. {clean_sentence_for_display(item['text'])}"
        for i, item in enumerate(selected)
    )

    # Prompt Generalization: Áp dụng được cho mọi thể loại văn bản
    system = (
        "Bạn là một biên tập viên chuyên nghiệp. "
        "Nhiệm vụ của bạn là TỔNG HỢP ĐẦY ĐỦ VÀ CHI TIẾT các câu trích xuất thành một văn bản trôi chảy. "
        "Không viết tiếng Trung, không Markdown."
    )

    user = f"""\
Dựa vào các câu bằng chứng (đã được trích xuất tự động từ tài liệu gốc) dưới đây, hãy tổng hợp thành một bài viết CHI TIẾT có độ dài xấp xỉ {target_words} từ.

Yêu cầu cực kỳ quan trọng để đạt đủ số từ:
1. KHÔNG TÓM TẮT QUÁ NGẮN: Tuyệt đối không được nén thông tin quá mức. Bạn phải cố gắng phát triển các ý dựa trên bằng chứng để bài viết dài gần chạm mức {target_words} từ.
2. GIỮ LẠI SỰ PHONG PHÚ CỦA CHI TIẾT: Hãy giữ lại và diễn đạt mềm mại các chi tiết cụ thể có trong bằng chứng (ví dụ: tên riêng, địa danh, mốc thời gian, số liệu, danh từ chuyên môn). Đừng gộp chúng lại quá ngắn gọn.
3. PHÁT TRIỂN Ý TRÔI CHẢY: Thay vì chỉ nối câu, hãy dùng các từ nối, câu chuyển đoạn để làm rõ bối cảnh và logic của vấn đề sao cho văn bản mạch lạc và có chiều sâu.
4. KHÔNG SAO CHÉP CƠ HỌC: Trình bày văn xuôi chia 4-5 đoạn rõ ràng, không gạch đầu dòng. TUYỆT ĐỐI KHÔNG tự bịa thêm thông tin ngoài bằng chứng.

TIÊU ĐỀ TÀI LIỆU:
{title}

CÁC CÂU BẰNG CHỨNG (Dùng để tổng hợp):
{evidence}

BẢN TỔNG HỢP CHI TIẾT:"""

    prompt = (
        f"<|im_start|>system\n{system}<|im_end|>\n"
        f"<|im_start|>user\n{user}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

    inputs = sum_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=7600,
    ).to("cuda")

    kwargs = dict(
        **inputs,
        max_new_tokens=1000,
        
        # 1. Bật lấy mẫu (sampling) nhưng siết chặt nhiệt độ
        do_sample=True,        
        temperature=0.15,      # Nhiệt độ rất thấp (0.15): Ép model cực kỳ bảo thủ, chỉ dùng đúng từ có trong context, KHÔNG chém gió số liệu.
        top_p=0.85,            # Chỉ lấy các từ có xác suất đúng cao nhất.
        
        # 2. Xóa bỏ các hình phạt ép model phải "sáng tạo" sai sự thật
        repetition_penalty=1.0, # 1.0 nghĩa là TẮT phạt lặp từ. Cho phép model thoải mái gọi đúng tên "đạo Thừa tuyên" hay "nhà Mạc" n lần mà không sợ sai.
        # ĐÃ XÓA HOÀN TOÀN: no_repeat_ngram_size=6 
        
        pad_token_id=sum_tokenizer.pad_token_id or sum_tokenizer.eos_token_id,
        eos_token_id=sum_tokenizer.eos_token_id,
    )

    bad_words_ids = get_cjk_bad_words_ids()
    if bad_words_ids:
        kwargs["bad_words_ids"] = bad_words_ids

    with torch.inference_mode():
        out = sum_model.generate(**kwargs)

    new_ids = out[0][inputs["input_ids"].shape[1]:]
    generated = sum_tokenizer.decode(new_ids, skip_special_tokens=True)
    return strip_bad_output(generated)

def looks_bad_polish(text: str, selected: list[dict], target_words: int) -> tuple[bool, list[str]]:
    reasons = []
    wc = word_count(text)

    # Nới lỏng Guardrail để không bắt lỗi oan bài dài/ngắn hợp lý
    min_ok = max(80, int(target_words * 0.50))
    max_ok = int(target_words * 1.50)
    
    if wc < min_ok:
        reasons.append(f"quá ngắn ({wc}/{target_words} từ)")
    if wc > max_ok:
        reasons.append(f"quá dài ({wc}/{target_words} từ)")
        
    # Không chặn Hán Việt/CJK vì rất dễ dính ở các tài liệu Sử học, Văn học
    # if _has_cjk(text):
    #     reasons.append("còn ký tự CJK/tiếng Trung")
        
    if re.search(r"(?m)^\s*(?:\d+\.|[-*])\s+", text):
        reasons.append("bị biến thành danh sách")
        
    return bool(reasons), reasons

def display_textrank_report(summary, out_path: Optional[str] = None) -> None:
    from IPython.display import HTML, display
    final_paragraphs = [
        p.strip()
        for p in re.split(r"\n\s*\n", summary.final)
        if p.strip()
    ]
    if not final_paragraphs:
        final_paragraphs = [summary.final]
    body_html = "\n".join(
        f"<p>{html.escape(p)}</p>"
        for p in final_paragraphs
    )
    rows = []
    for rank, item in enumerate(summary.selected, 1):
        sent = clean_sentence_for_display(item["text"])
        rows.append(
            "<tr>"
            f"<td>{rank:02d}</td>"
            f"<td>{item['score']:.5f}</td>"
            f"<td>{item['word_count']}</td>"
            f"<td>{html.escape(sent)}</td>"
            "</tr>"
        )
    status = "Qwen polish" if summary.used_llm else "TextRank extractive"
    fallback = "Có" if summary.used_fallback else "Không"
    path_html = f"<div class='path'>Đã lưu: <code>{html.escape(out_path)}</code></div>" if out_path else ""

    display(HTML(f"""
    <style>
      .tr-report {{
        font-family: Arial, sans-serif;
        max-width: 980px;
        border: 1px solid #d9e2ec;
        border-radius: 10px;
        padding: 18px 20px;
        background: #ffffff;
        color: #17202a;
        line-height: 1.58;
      }}
      .tr-report h2 {{
        margin: 0 0 8px 0;
        font-size: 22px;
        color: #12355b;
      }}
      .tr-report .meta {{
        display: flex;
        flex-wrap: wrap;
        gap: 8px;
        margin: 10px 0 16px 0;
      }}
      .tr-report .chip {{
        background: #eef6ff;
        border: 1px solid #cfe4ff;
        border-radius: 999px;
        padding: 4px 10px;
        font-size: 13px;
      }}
      .tr-report .summary {{
        border-left: 4px solid #2f80ed;
        padding: 2px 0 2px 14px;
        margin-top: 10px;
      }}
      .tr-report .summary p {{
        margin: 0 0 12px 0;
        text-align: justify;
      }}
      .tr-report details {{
        margin-top: 16px;
      }}
      .tr-report summary {{
        cursor: pointer;
        font-weight: 700;
        color: #12355b;
      }}
      .tr-report table {{
        border-collapse: collapse;
        width: 100%;
        margin-top: 10px;
        font-size: 13px;
      }}
      .tr-report th, .tr-report td {{
        border: 1px solid #e2e8f0;
        padding: 7px 8px;
        vertical-align: top;
      }}
      .tr-report th {{
        background: #f6f8fb;
      }}
      .tr-report .path {{
        margin-top: 12px;
        font-size: 13px;
        color: #52616b;
      }}
    </style>
    <div class="tr-report">
      <h2>Bản Tổng Hợp TextRank</h2>
      <div><b>{html.escape(summary.title)}</b></div>
      <div class="meta">
        <span class="chip">Số từ: {word_count(summary.final):,}</span>
        <span class="chip">Câu TextRank: {summary.sentence_count:,}</span>
        <span class="chip">Câu chọn: {len(summary.selected):,}</span>
        <span class="chip">Chế độ: {status}</span>
        <span class="chip">Fallback: {fallback}</span>
        <span class="chip">Thời gian: {summary.elapsed_seconds:.1f}s</span>
      </div>
      <div class="summary">
        {body_html}
      </div>
      <details>
        <summary>Xem các câu TextRank đã chọn</summary>
        <table>
          <thead>
            <tr>
              <th>#</th>
              <th>Score</th>
              <th>Từ</th>
              <th>Câu</th>
            </tr>
          </thead>
          <tbody>
            {''.join(rows)}
          </tbody>
        </table>
      </details>
      {path_html}
    </div>
    """))

@dataclass
class TextRankSummaryResult:
    title: str
    body_word_count: int
    sentence_count: int
    selected: list[dict]
    extractive: str
    final: str
    used_llm: bool
    used_fallback: bool
    guard_reasons: list[str]
    elapsed_seconds: float

def summarize_pdf_by_textrank(
    raw_text: str,
    target_words: int = 500,
    use_llm_polish: bool = True,
    verbose: bool = True,
) -> TextRankSummaryResult:
    t0 = time.time()
    title = extract_title(raw_text)
    body_text = clean_pdf_body_text(raw_text)
    
    # [QUAN TRỌNG] Tự động nhân hệ số số từ cần trích xuất lên 1.6 lần
    # để cung cấp dư dả nguyên liệu cho LLM viết bài dài
    extract_target = int(target_words * 1.6) if use_llm_polish else target_words
    
    selected, sentences, scores = textrank_extract(body_text, target_words=extract_target)
    extractive = extractive_summary_from_selected(selected)

    if verbose:
        print(f"Tiêu đề: {title}")
        print(f"Độ dài thân bài sau lọc: {word_count(body_text):,} từ")
        print(f"Số câu đưa vào TextRank: {len(sentences):,}")
        print(f"Số câu được chọn: {len(selected):,}")
        print(f"Bản TextRank extractive (nguyên liệu thô): {word_count(extractive):,} từ")

    final = extractive
    used_llm = False
    used_fallback = False
    guard_reasons = []

    if use_llm_polish:
        print("Đang cho Qwen tổng hợp các câu TextRank...")
        try:
            candidate = qwen_polish_textrank(title, selected, target_words=target_words)
            bad, reasons = looks_bad_polish(candidate, selected, target_words)
            if bad:
                print("Output Qwen không đạt guard, dùng bản TextRank extractive.")
                print("Lý do:", " | ".join(reasons))
                used_fallback = True
                guard_reasons = reasons
            else:
                final = candidate
                used_llm = True
        except Exception as e:
            print(f"Qwen lỗi, dùng bản TextRank extractive: {e}")
            used_fallback = True
            guard_reasons = [str(e)]

    return TextRankSummaryResult(
        title=title,
        body_word_count=word_count(body_text),
        sentence_count=len(sentences),
        selected=selected,
        extractive=extractive,
        final=_squeeze(final),
        used_llm=used_llm,
        used_fallback=used_fallback,
        guard_reasons=guard_reasons,
        elapsed_seconds=time.time() - t0,
    )

In [24]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 17 — Chạy TextRank tóm tắt bài báo 168.pdf        ║
# ╚══════════════════════════════════════════════════════════╝

PDF_CANDIDATES = [
    "/kaggle/input/datasets/vietnguyencong123/dataops2/168.pdf",
    "/kaggle/input/dataops2/168.pdf",
    "/kaggle/working/168.pdf",
    "168.pdf",
]

PDF_PATH = next((p for p in PDF_CANDIDATES if os.path.exists(p)), None)
if PDF_PATH is None:
    raise FileNotFoundError("Không tìm thấy file pdf. Hãy sửa lại PDF_CANDIDATES/PDF_PATH.")

def ask_int(prompt: str, default: int, min_value: int = 100, max_value: int = 3000) -> int:
    raw = input(f"{prompt} [{default}]: ").strip()
    if not raw:
        return default
    try:
        value = int(raw)
    except ValueError:
        print(f"Giá trị không hợp lệ, dùng mặc định {default}.")
        return default
    if value < min_value:
        print(f"Số từ quá thấp, dùng {min_value}.")
        return min_value
    if value > max_value:
        print(f"Số từ quá cao, dùng {max_value}.")
        return max_value
    return value


def ask_yes_no(prompt: str, default: bool = True) -> bool:
    default_text = "y" if default else "n"
    raw = input(f"{prompt} (y/n) [{default_text}]: ").strip().lower()
    if not raw:
        return default
    return raw in {"y", "yes", "1", "true", "co", "có"}


TARGET_WORDS = ask_int(
    "Nhập số từ tóm tắt mong muốn",
    default=500,
    min_value=150,
    max_value=2000,
)
USE_LLM_POLISH = ask_yes_no(
    "Dùng Qwen để biên tập lại câu TextRank?",
    default=True,
)

print(f"Yêu cầu độ dài: khoảng {TARGET_WORDS:,} từ")
print(f"Qwen polish: {USE_LLM_POLISH}")

print(f"PDF: {PDF_PATH}")
raw_text = get_pdf_text_smart(PDF_PATH, cfg)

summary = summarize_pdf_by_textrank(
    raw_text=raw_text,
    target_words=TARGET_WORDS,
    use_llm_polish=USE_LLM_POLISH,
    verbose=True,
)

out_path = "/kaggle/working/summary_textrank.txt"
with open(out_path, "w", encoding="utf-8") as f:
    f.write(f"Tiêu đề: {summary.title}\n")
    f.write(f"Số từ thân bài sau lọc: {summary.body_word_count}\n")
    f.write(f"Số câu TextRank: {summary.sentence_count}\n")
    f.write(f"Số câu chọn: {len(summary.selected)}\n")
    f.write(f"Dùng Qwen polish: {summary.used_llm}\n")
    f.write(f"Dùng fallback TextRank: {summary.used_fallback}\n")
    if summary.guard_reasons:
        f.write("Guard reasons: " + " | ".join(summary.guard_reasons) + "\n")
    f.write("=" * 70 + "\n\n")
    f.write("BẢN TÓM TẮT CUỐI\n")
    f.write("=" * 70 + "\n")
    f.write(summary.final + "\n\n")
    f.write("=" * 70 + "\n")
    f.write("BẢN TEXT RANK EXTRACTIVE\n")
    f.write("=" * 70 + "\n")
    f.write(summary.extractive + "\n\n")
    f.write("=" * 70 + "\n")
    f.write("CÂU TEXT RANK ĐÃ CHỌN\n")
    f.write("=" * 70 + "\n")
    for rank, item in enumerate(summary.selected, 1):
        clean_sent = clean_sentence_for_display(item["text"])
        f.write(f"{rank:02d}. [sent={item['idx']}, score={item['score']:.6f}] {clean_sent}\n")

display_textrank_report(summary, out_path=out_path)

Nhập số từ tóm tắt mong muốn [500]:  500
Dùng Qwen để biên tập lại câu TextRank? (y/n) [y]:  y


Yêu cầu độ dài: khoảng 500 từ
Qwen polish: True
PDF: /kaggle/input/datasets/vietnguyencong123/dataops2/168.pdf
───────────────────────────────────────────────────────
  File       : 168.pdf
  Kết quả    : ✅ No OCR needed
  Độ tin cậy : high
───────────────────────────────────────────────────────
  Tổng trang          : 11 (phân tích: 11)
  Ký tự TB/trang      : 2588.0
  Tỷ lệ trang có text : 100.0%
  Font nhúng          : Có
  Trang bị ảnh phủ    : 1
  Tỷ lệ ký tự rác     : 3.7%
───────────────────────────────────────────────────────
  • [OK] Ký tự TB/trang đủ lớn (2588).
  • [OK] 100% trang có văn bản.
  • [OK] Có 1 trang ảnh lớn nhưng không chiếm đa số.
  • [OK] Font nhúng đầy đủ.
  • [OK] Ký tự rác thấp (3.7%).
  • [OK] Encoding Unicode chuẩn, text đọc được bình thường.
───────────────────────────────────────────────────────
⚡ Dùng PyMuPDF direct (không cần OCR)
Tiêu đề: Phân tích thực trạng sử dụng thuốc điều trị tăng huyết áp tại
Độ dài thân bài sau lọc: 3,769 từ
Số câu đưa vào Te

#,Score,Từ,Câu
01,0.01207,21,Chức năng thận của bệnh nhân được đánh giá dựa vào độ thanh thải Creatinine (Clcr) tính theo công thức Cockroft-Gault.
02,0.02272,85,"Đặc điểm của bệnh nhân Đặc điểm Số bệnh nhân (N = 400) Tỷ lệ (%) 18 – 69 213 53,25 70 – 79 133 33,25 Tuổi Trung bình 69,12 ± 9,15 tuổi Giới Rối loạn lipid máu 357 89,25 Bệnh mắc kèm Cơn đau thắt ngực 243 60,5 Trong số các bệnh lý mắc kèm, bệnh nhân THA kèm rối loạn lipid máu chiếm tỷ lệ nhiều nhất (89,25%), bệnh nhân THA kèm cơn đau thắt ngực có tỷ lệ thấp hơn là 60,5%."
03,0.02141,85,"Tỷ lệ chỉ định sử dụng các nhóm thuốc điều trị THA STT Các nhóm thuốc Số đơn (N = 1600) Tỷ lệ (%) 1 Lợi tiểu (LT) 1067 66,69 2 Chẹn thụ thể angiotensin (CTTA) 1003 62,69 3 Ức chế men chuyển (ƯCMC) 566 35,38 4 Chẹn kênh canxi (CKCa) 261 16,31 5 Chẹn beta (CB) 235 14,69 6 Chủ vận chọn lọc alpha-2 giao cảm 2 0,13 Nhóm thuốc LT (66,69%) và CTTA (62,69%) là hai nhóm thuốc được kê đơn nhiều nhất."
04,0.02239,28,"Nhóm ≥ 70 tuổi có tỷ lệ đạt huyết áp mục tiêu thấp hơn một chút so với hai nhóm còn lại, dao động từ 95,7% đến 96,3%."
05,0.02165,18,"Ở nhóm bệnh nhân THA kèm đau thắt ngực, tỷ lệ đơn thuốc phù hợp chỉ đạt 16,15%."
06,0.02220,84,"Độ tuổi trung bình khá cao khoảng 69,12 ± 9,15 tuổi, cao hơn so với nghiên cứu của Ngô Thị Hải Yến tại Trung tâm y tế huyện Quế Phong, tỉnh Nghệ An năm 2023 và của Chúc Thị Hà năm 2024 tại Trung tâm y tế huyện Chiêm Hóa, tỉnh Tuyên Quang Có 36 phác đồ được sử dụng trong mẫu nghiên cứu, trong đó: Nhóm đơn trị liệu (phác đồ 1 thuốc) được sử dụng nhiều nhất là nhóm ƯCMC với 8,56%."
07,0.02334,22,"Phác đồ phối hợp 3 thuốc được sử dụng nhiều nhất là nhóm thuốc ƯCMC + CTTA + lợi tiểu, với 3,81%."
08,0.02172,49,"Trong mẫu nghiên cứu có nhiều phác đồ sử dụng 2 CTTA, 2 ƯCMC và dùng đồng thời CTTA + ƯCMC trong cùng 1 đơn thuốc, đây là những phối hợp cần tránh do có thể gây ra sự cạnh tranh thụ thể cũng như tăng tác dụng phụ."
09,0.03335,72,"Phân tích tính hợp lý trong việc sử dụng thuốc điều trị THA tại Phòng khám huyết áp ngoại trú, TTYT huyện Văn Giang Có 243/400 bệnh nhân trong mẫu nghiên cứu có chẩn đoán là cơn đau thắt ngực, với 972 đơn thuốc trong 4 tháng theo dõi, số đơn thuốc được kê đơn phù hợp với phác đồ là 157 đơn, chỉ chiếm 16,15% - một tỷ lệ khá thấp."
10,0.02369,61,"Có 157/400 bệnh nhân với 628 đơn thuốc trong 4 tháng theo dõi ở bệnh nhân THA không có các bệnh lý mắc kèm, nhóm bệnh nhân này thì phác đồ phù hợp là ƯCMC hoặc CTTA + CKCa/LT; ƯCMC hoặc CTTA + CKCa + LT; THA kháng trị là ƯCMC hoặc CTTA + CKCa + LT + MRA/LT khác/CB/chẹn alpha."


## Pipeline

PDF

-> đọc text theo cột bằng PyMuPDF

-> lọc header/footer/DOI/tài liệu tham khảo

-> bỏ phần TÓM TẮT/ABSTRACT nếu PDF có sẵn

-> tách văn bản thân bài thành từng câu

-> tính độ giống nhau giữa các câu bằng TF-IDF cosine similarity

-> chạy PageRank trên graph câu

-> chọn các câu điểm cao theo ngân sách ~500 từ

-> dọn câu bị vỡ layout/bảng

-> nếu USE_LLM_POLISH=True thì đưa các câu TextRank đã chọn cho Qwen biên tập lại

-> nếu Qwen sinh lỗi thì fallback về bản TextRank extractive
